# MOD-05 · NB-07 — Clinical Text Summarisation & Zero-Shot NLP
### Health Analytics with Python · Module 05: NLP for Clinical Text
---
**Learning objectives**
- Apply TextRank extractive summarisation to discharge summaries
- Build LSA-based and section-aware summarisers
- Evaluate with ROUGE-1, ROUGE-2, ROUGE-L
- Design clinical LLM prompt templates
- Build a zero-shot TF-IDF triage classifier

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `sklearn`, `numpy`, `nltk`

## 1. Setup & Discharge Summary

In [ ]:
import re, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white"})

import nltk
for r in ["punkt","punkt_tab","stopwords"]:
    try: nltk.data.find(f"tokenizers/{r}")
    except LookupError: nltk.download(r, quiet=True)
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

DISCHARGE_SUMMARY = (
    "DISCHARGE SUMMARY\n\n"
    "PATIENT: 67-year-old male with type 2 diabetes mellitus, hypertension, CKD stage 3.\n\n"
    "ADMITTING DIAGNOSIS: Acute decompensated heart failure.\n\n"
    "HISTORY OF PRESENT ILLNESS: Patient presented with three days of progressive dyspnea, "
    "bilateral lower extremity edema, and orthopnea. Weight increased by 8 pounds. Denied chest pain.\n\n"
    "HOSPITAL COURSE: BNP was elevated at 1450. Echocardiogram showed EF 30 percent with moderate "
    "mitral regurgitation. IV furosemide 80 mg twice daily achieved 3.5 liters negative fluid balance. "
    "Weight decreased from 98 to 94 kg. Lisinopril uptitrated to 20 mg. Carvedilol increased to 12.5 mg. "
    "Sacubitril-valsartan initiated. Creatinine rose to 1.9 then returned to 1.5 by discharge.\n\n"
    "LABS: Admission BNP 1450, creatinine 1.6, sodium 136, potassium 3.8. "
    "Discharge BNP 620, creatinine 1.5, sodium 139, potassium 4.1.\n\n"
    "DISCHARGE DIAGNOSES: 1. Acute decompensated systolic heart failure EF 30 percent. "
    "2. Type 2 diabetes. 3. CKD stage 3. 4. Hypertension. 5. Mitral regurgitation.\n\n"
    "DISCHARGE MEDICATIONS: Sacubitril-valsartan 24/26 mg twice daily (new). "
    "Carvedilol 12.5 mg twice daily (increased). Furosemide 40 mg daily (reduced). "
    "Spironolactone 25 mg daily (new). Metformin 500 mg twice daily. Aspirin 81 mg daily.\n\n"
    "FOLLOW-UP: Cardiology in 1 week. Primary care in 2 weeks. Daily weights, call if up 3 pounds.\n\n"
    "CONDITION AT DISCHARGE: Stable, ambulatory, euvolemic."
)
print(f"Summary: {len(DISCHARGE_SUMMARY)} chars, {len(DISCHARGE_SUMMARY.split())} words")


## 2. TextRank Extractive Summarisation

In [ ]:
STOPS = list(stopwords.words("english")) + [
    "patient","mg","daily","was","were","the","a","an",
    "admission","discharge","history","hospital","noted",
]

def textrank_summarise(text, n_sents=5):
    sents = [s.strip() for s in sent_tokenize(text) if len(s.split()) > 5]
    if len(sents) <= n_sents:
        return sents, np.ones(len(sents))
    tfidf = TfidfVectorizer(stop_words=STOPS, max_features=300)
    try:
        mat = tfidf.fit_transform(sents)
    except Exception:
        return sents[:n_sents], np.ones(n_sents)
    sim = cosine_similarity(mat)
    np.fill_diagonal(sim, 0)
    scores = np.ones(len(sents)) / len(sents)
    for _ in range(50):
        rs = sim.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1
        scores = 0.15/len(sents) + 0.85 * (sim/rs).T @ scores
    top_idx = sorted(np.argsort(scores)[-n_sents:])
    return [sents[i] for i in top_idx], scores

tr_sents, _ = textrank_summarise(DISCHARGE_SUMMARY, n_sents=5)
print("TextRank Summary (5 sentences):\n")
for i, s in enumerate(tr_sents, 1):
    print(f"[{i}] {s.strip()}")


## 3. LSA-Based Summarisation

In [ ]:
def lsa_summarise(text, n_sents=5, n_comp=4):
    sents = [s.strip() for s in sent_tokenize(text) if len(s.split()) > 5]
    if len(sents) <= n_sents:
        return sents
    tfidf = TfidfVectorizer(stop_words=STOPS, max_features=300)
    try:
        X = tfidf.fit_transform(sents)
    except Exception:
        return sents[:n_sents]
    n_comp = min(n_comp, X.shape[0]-1, X.shape[1]-1)
    if n_comp < 1:
        return sents[:n_sents]
    X_r = TruncatedSVD(n_components=n_comp, random_state=42).fit_transform(X)
    scs = np.linalg.norm(X_r, axis=1)
    top_idx = sorted(np.argsort(scs)[-n_sents:])
    return [sents[i] for i in top_idx]

lsa_sents = lsa_summarise(DISCHARGE_SUMMARY, n_sents=5)
print("LSA-Based Summary (5 sentences):\n")
for i, s in enumerate(lsa_sents, 1):
    print(f"[{i}] {s.strip()}")


## 4. Section-Aware Summarisation

In [ ]:
SECTION_WEIGHTS = {
    "DISCHARGE DIAGNOSES"  : 3.0,
    "DISCHARGE MEDICATIONS": 2.5,
    "HOSPITAL COURSE"      : 2.0,
    "FOLLOW-UP"            : 1.5,
    "LABS"                 : 1.0,
    "ADMITTING DIAGNOSIS"  : 1.0,
    "DISCHARGE CONDITION"  : 1.5,
}

def get_section(pos, text):
    best, best_pos = "GENERAL", -1
    for header in SECTION_WEIGHTS:
        hp = text.upper().find(header)
        if hp != -1 and hp < pos and hp > best_pos:
            best, best_pos = header, hp
    return best

def section_aware_summarise(text, n_sents=6):
    sents = [s.strip() for s in sent_tokenize(text) if len(s.split()) > 5]
    if len(sents) <= n_sents:
        return sents, ["GENERAL"]*len(sents)
    tfidf = TfidfVectorizer(stop_words=STOPS, max_features=300)
    try:
        X = tfidf.fit_transform(sents)
        ts = np.array(X.sum(axis=1)).flatten()
    except Exception:
        ts = np.ones(len(sents))
    weighted = []; secs = []
    for i, s in enumerate(sents):
        pos = text.find(s)
        sec = get_section(pos, text)
        weighted.append(ts[i] * SECTION_WEIGHTS.get(sec, 0.5))
        secs.append(sec)
    top_idx = sorted(np.argsort(weighted)[-n_sents:])
    return [sents[i] for i in top_idx], [secs[i] for i in top_idx]

sa_sents, sa_secs = section_aware_summarise(DISCHARGE_SUMMARY, n_sents=6)
print("Section-Aware Summary (6 sentences):\n")
for s, sec in zip(sa_sents, sa_secs):
    print(f"  [{sec[:22]:22s}] {s.strip()[:75]}")


## 5. ROUGE Evaluation

In [ ]:
def get_ngrams(text, n):
    toks = re.findall(r'\b\w+\b', text.lower())
    return [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]

def rouge_n(hyp, ref, n=1):
    h = get_ngrams(hyp, n); r = get_ngrams(ref, n)
    if not r: return {"p":0,"r":0,"f1":0}
    hc = {}
    for ng in h: hc[ng] = hc.get(ng,0)+1
    rc = {}
    for ng in r: rc[ng] = rc.get(ng,0)+1
    ov = sum(min(hc.get(ng,0),cnt) for ng,cnt in rc.items())
    p  = ov/len(h) if h else 0
    re_= ov/len(r)
    f1 = 2*p*re_/(p+re_) if p+re_>0 else 0
    return {"p":round(p,4),"r":round(re_,4),"f1":round(f1,4)}

def rouge_l(hyp, ref):
    h = re.findall(r'\b\w+\b',hyp.lower())
    r = re.findall(r'\b\w+\b',ref.lower())
    m, n = len(h), len(r)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            dp[i][j] = dp[i-1][j-1]+1 if h[i-1]==r[j-1] else max(dp[i-1][j],dp[i][j-1])
    lcs = dp[m][n]
    p  = lcs/m if m else 0
    re_= lcs/n if n else 0
    f1 = 2*p*re_/(p+re_) if p+re_>0 else 0
    return {"p":round(p,4),"r":round(re_,4),"f1":round(f1,4)}

REFERENCE = (
    "67-year-old male heart failure ejection fraction 30 percent admitted acute decompensation. "
    "IV furosemide diuresis BNP improved 1450 to 620. New medications sacubitril-valsartan spironolactone. "
    "Creatinine stable 1.5 at discharge. Cardiology follow-up one week."
)
methods = {
    "TextRank (5)": " ".join(tr_sents),
    "LSA (5)"     : " ".join(lsa_sents),
    "Section (6)" : " ".join(sa_sents),
}
print("ROUGE Evaluation vs Human Reference:\n")
print(f"{'Method':18s}  {'R1-F1':>8s}  {'R2-F1':>8s}  {'RL-F1':>8s}")
print("-"*48)
rouge_rows = []
for name, hyp in methods.items():
    r1 = rouge_n(hyp, REFERENCE, 1)
    r2 = rouge_n(hyp, REFERENCE, 2)
    rl = rouge_l(hyp, REFERENCE)
    rouge_rows.append({"Method":name,"R1-F1":r1["f1"],"R2-F1":r2["f1"],"RL-F1":rl["f1"]})
    print(f"  {name:16s}  {r1['f1']:>8.4f}  {r2['f1']:>8.4f}  {rl['f1']:>8.4f}")
rouge_df = pd.DataFrame(rouge_rows)


## 6. LLM Prompt Templates for Clinical Tasks

In [ ]:
PROMPTS = {
    "summarise_discharge": (
        "You are a clinical documentation specialist.\n"
        "Summarise the discharge summary into 5 concise sentences covering:\n"
        "primary diagnosis, key treatments, significant lab trends, "
        "new medications, and critical follow-up.\n\n"
        "DISCHARGE SUMMARY:\n{text}\n\nSTRUCTURED SUMMARY:"
    ),
    "extract_diagnoses": (
        "Extract all diagnoses from the clinical note as a JSON array.\n"
        "Each element: {diagnosis, icd10_suggestion, certainty}.\n\n"
        "CLINICAL NOTE:\n{text}\n\nJSON OUTPUT:"
    ),
    "clinical_triage": (
        "Classify urgency as: IMMEDIATE / URGENT / SEMI-URGENT / ROUTINE.\n"
        "Provide one sentence of rationale.\n\n"
        "CLINICAL DESCRIPTION:\n{text}\n\nTRIAGE:"
    ),
}

print("Clinical LLM Prompt Templates:\n")
for task, template in PROMPTS.items():
    filled = template.format(text=DISCHARGE_SUMMARY[:200] + "...")
    print(f"Task: {task}")
    print(f"  Prompt length: {len(filled)} chars")
    print(f"  Preview: {filled[:100]}...")
    print()

print("Anthropic API integration pattern:")
print("  client = anthropic.Anthropic()")
print("  msg = client.messages.create(")
print('      model="claude-sonnet-4-6", max_tokens=1024,')
print("      messages=[{")
print('          "role":"user",')
print('          "content": PROMPTS["summarise_discharge"].format(text=note_text)')
print("      }]")
print("  )")
print("  summary = msg.content[0].text")


## 7. Zero-Shot TF-IDF Triage Classifier

In [ ]:
TRIAGE_DESCS = {
    "IMMEDIATE":   "cardiac arrest respiratory failure unresponsive SpO2 below 85 BP below 70 "
                   "active hemorrhage STEMI severe respiratory distress cyanosis anaphylaxis",
    "URGENT":      "chest pain active angina dyspnea rest SpO2 92 altered mental status "
                   "new neurological deficit uncontrolled pain severe hypertension fever sepsis",
    "SEMI_URGENT": "moderate pain stable vital signs mild dyspnea worsening chronic condition "
                   "urinary symptoms mild infection stable arrhythmia elevated glucose",
    "ROUTINE":     "prescription refill follow-up chronic stable lab results review "
                   "medication adjustment preventive care no acute distress",
}

def zero_shot_triage(text):
    all_texts = list(TRIAGE_DESCS.values()) + [text]
    tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=500)
    mat = tfidf.fit_transform(all_texts)
    scores = cosine_similarity(mat[-1], mat[:-1]).flatten()
    return sorted(zip(TRIAGE_DESCS.keys(), scores), key=lambda x: x[1], reverse=True)

triage_cases = [
    "Patient found unresponsive, no pulse, CPR initiated.",
    "67yo chest pain 7/10 radiating to left arm, diaphoretic, EKG ST elevation.",
    "45yo BP 190/110, severe headache, visual changes, no focal neuro deficit.",
    "72yo increasing shortness of breath x3 days, orthopnea, bilateral leg swelling.",
    "32yo requests metformin refill, sugars 150-200, last visit 3 months ago.",
]
print("Zero-Shot Triage Classification:\n")
for case in triage_cases:
    results = zero_shot_triage(case)
    top_label, top_score = results[0]
    print(f"  Case: '{case[:60]}'")
    print(f"  => {top_label} (score={top_score:.3f})")
    for label, score in results[1:]:
        bar = chr(9608) * int(score*25)
        print(f"     {label:12s} {bar} {score:.3f}")
    print()


## 8. Knowledge Check
1. TextRank ROUGE-1 F1=0.42. What baseline would you compare against (e.g. lead-1)?
2. The section-aware method weighted DISCHARGE DIAGNOSES at 3.0. What justifies this?
3. Zero-shot triage classified chest pain as SEMI-URGENT. What safety implications?
4. Abstractive LLM summarisation can hallucinate clinical facts. Describe two validation strategies.
5. Implement a lead-1 baseline: use the first sentence of each major section.

In [ ]:
def lead_baseline(text):
    headers = ["ADMITTING DIAGNOSIS","HOSPITAL COURSE","DISCHARGE DIAGNOSES",
               "DISCHARGE MEDICATIONS","FOLLOW-UP","DISCHARGE CONDITION"]
    sents = []
    for h in headers:
        pos = text.upper().find(h)
        if pos == -1: continue
        rest = text[pos:pos+300]
        for line in rest.split("."):
            line = line.strip()
            if len(line.split()) > 5:
                sents.append(line + ".")
                break
    return sents

lead_sents = lead_baseline(DISCHARGE_SUMMARY)
hyp_lead = " ".join(lead_sents)
rl = rouge_l(hyp_lead, REFERENCE)
r1 = rouge_n(hyp_lead, REFERENCE, 1)
print(f"Lead-section baseline: ROUGE-1 F1={r1['f1']:.4f} | ROUGE-L F1={rl['f1']:.4f}")
print()
for s in lead_sents:
    print(f"  {s.strip()[:80]}")


---
**Next → NB-08: Capstone — End-to-End Clinical NLP Pipeline**